# 모듈 4. 벡터화와 RAG, 그리고 Claude 연결

**목표**
1. 공시 원문을 **청크**로 나누고, 반복되는 **공통 문구**를 분리한다.
2. 로컬 임베딩으로 벡터화해 DuckDB에 **벡터 + 키워드(BM25) 하이브리드 검색**을 만든다.
3. "무엇을 RAG로, 무엇을 SQL로 답할 것인가"를 실험으로 확인한다.
4. 상품 DB와 검색을 **MCP 서버**로 Claude Desktop에 연결한다.

**산출물**: `data/aifinance.duckdb`의 `chunks` 테이블(색인), `src/aifinance/mcp_server.py`

## 1. 설계 선택

| 선택 | 결정 | 이유 |
|---|---|---|
| 벡터 저장소 | **DuckDB** (별도 벡터 DB 없음) | 상품 DB와 같은 파일. SQL 필터와 검색을 한 쿼리에서 결합. 수만 청크는 전수 비교로도 충분히 빠름 |
| 임베딩 | **fastembed + `BAAI/bge-small-en-v1.5`** (67MB) | 로컬·무료·torch 불필요. 공시가 영어이고, MCP 구조에서는 Claude가 검색어를 영어로 바꿔 주므로 영어 모델로 충분 |
| 검색 | **하이브리드** (벡터 + BM25 → RRF 융합) | 벡터는 뜻이 비슷한 문단, BM25는 `Trigger Value`·CUSIP 같은 정확한 용어에 강함 |
| 청킹 | 문단 단위, 최대 1,200자, **상품 정보 머리글** 부착 | 청크만 봐도 어느 상품인지 알 수 있어 검색·인용이 쉬워짐 |

> 한국어로 직접 검색하려면 다국어 모델(`rag.MULTILINGUAL_MODEL`)을 쓸 수 있습니다. 실험해 보면 색인 시간이 약 3~10배 늘고, 고유명사(예: "엔비디아")는 여전히 약합니다.

In [1]:
import json
import sys

import duckdb
import pandas as pd

from aifinance.config import DATA_DIR, INTERIM_DIR, PROJECT_ROOT
from aifinance.db import DB_PATH
from aifinance.rag import (DEFAULT_MODEL, find_boilerplate, index_database, make_chunks, paragraphs, search)

pd.set_option("display.max_colwidth", 160)
docs = pd.read_parquet(INTERIM_DIR / "documents.parquet")
texts = docs.dropna(subset=["text"]).set_index("accessionNumber").text
con = duckdb.connect(str(DB_PATH))

## 2. 공시의 절반은 공통 문구다

424B2에는 위험 고지, 세금, 판매 방식 같은 **모든 상품에 거의 똑같이 들어가는 문단**이 많습니다.
이를 상품별 청크와 섞어 색인하면 어떤 질문을 해도 같은 공통 문단이 수십 번 검색됩니다.

방법: 문단마다 해시를 만들어, **표본 문서의 5% 이상(최소 5건)에 같은 문단이 나오면 공통 문구**로 봅니다.
공통 문구는 한 번만 색인하고, 상품별 청크에는 고유한 문단만 남깁니다.

> ⚠️ 처음에는 숫자를 지우고 해시를 만들었습니다(날짜만 다른 문단을 묶으려고). 그랬더니 `Trigger Value: 50.00%`처럼 **숫자만 다른 상품별 조건 줄**까지 공통 문구로 분류됐습니다. 지금은 페이지 번호(`PS-3`)만 지웁니다.

In [2]:
meta = con.sql('''
    SELECT t.accession_number, t.cusip, t.product_name, t.doc_status, d.filing_date
    FROM terms t JOIN documents d USING (accession_number) WHERE t.doc_status = 'final'
''').df()
meta["text"] = meta.accession_number.map(texts)

boiler = find_boilerplate(meta.text.tolist())
chunks = make_chunks(meta, boiler)

total = meta.text.str.len().sum()
indexed = chunks.text.str.len().sum()
print(f"확정본 {len(meta)}건, 원문 {total:,}자")
print(f"공통 문구로 분류된 문단 종류: {len(boiler):,}")
print(f"청크 {len(chunks):,}개 = 상품별 {(~chunks.is_boilerplate).sum():,} + 공통 {chunks.is_boilerplate.sum():,}")
print(f"색인 대상 {indexed:,}자 → 원문 대비 {indexed / total:.0%}")

확정본 150건, 원문 8,687,428자
공통 문구로 분류된 문단 종류: 701
청크 4,257개 = 상품별 3,556 + 공통 701
색인 대상 4,431,200자 → 원문 대비 51%


In [3]:
# 상품별 청크 예시: 머리글 + 고유 문단
print(chunks[~chunks.is_boilerplate].text.iloc[1][:900])

[Registration Statement Nos. 333-293684 and 333-293684-01 Dated October 9, 2026 Filed pursuant to Rule 424(b)(2) STRUCTURED INVESTMENTS Opportunities in U.S. Equities Contingent Income Auto-Callable Securities due October 14, 2027 | CUSIP 46661P5F0 | final | 2026-10-13]
In addition, if the closing price of the underlying stock is greater than or equal to the initial stock price on any determination date (other than the final determination date), the securities will be automatically redeemed for an amount per security equal to the stated principal amount plus the contingent quarterly payment with respect to that determination date (plus any previously unpaid contingent quarterly payments with respect to any prior determination dates). If the securities have not been automatically redeemed prior to maturity and the final stock price is greater than or equal to the downside threshold level,


## 3. 색인 만들기

청크를 임베딩해 `chunks` 테이블에 `FLOAT[384]` 벡터로 저장하고, DuckDB `fts` 확장으로 BM25 색인도 만듭니다.
처음 실행할 때 임베딩 모델(약 67MB)을 내려받습니다. 노트북 CPU에서 4천여 청크에 2~5분 걸립니다.

In [4]:
if con.execute("SELECT count(*) FROM information_schema.tables WHERE table_name = 'chunks'").fetchall()[0][0]:
    print("기존 색인 사용:", con.execute("SELECT model, dim, built_at FROM rag_meta").fetchall()[0])
else:
    index_database(con, texts, DEFAULT_MODEL)
con.sql("SELECT count(*) AS 청크, count_if(is_boilerplate) AS 공통문구 FROM chunks").df()

기존 색인 사용: ('BAAI/bge-small-en-v1.5', 384, datetime.datetime(2026, 11, 12, 12, 26, 32, 629272))


,청크,공통문구
0,4257,701.0


## 4. 검색 실험

### 4-1. 벡터 vs 키워드 vs 하이브리드

In [5]:
def show(q, mode="hybrid", k=3, **kw):
    r = search(con, q, k=k, mode=mode, **kw)
    print(f"\n[{mode}] {q}")
    for t in r.text:
        head, _, body = t.partition("\n")
        print(f"  · {head[:110]}\n      {body[:160]}")

q = "How is the estimated value of the notes determined?"
for mode in ["vector", "keyword", "hybrid"]:
    show(q, mode)

/Users/joocheol/Documents/GitHub/AIFinance/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



[vector] How is the estimated value of the notes determined?
  · [상품명 미상 | CUSIP 46661QKK0 | final | 2026-11-10]
      These models are dependent on inputs such as the traded market prices of comparable derivative instruments and on various other inputs, some of which are market
  · [상품명 미상 | CUSIP 46661PSY4 | final | 2026-10-23]
      The estimated value of the notes is determined by reference to internal pricing models of our affiliates when the terms of the notes are set. This estimated val
  · [상품명 미상 | CUSIP 46661PSY4 | final | 2026-10-23]
      The estimated value of the notes is only an estimate determined by reference to several factors. The original issue price of the notes exceeds the estimated val

[keyword] How is the estimated value of the notes determined?
  · [공통 문구 | JPMorgan Chase Financial 구조화 노트]
      The estimated value of the notes is only an estimate determined by reference to several factors. The original issue price of the
  · [상품명 미상 | CUSIP 46661PPF8 | final

In [6]:
show("issuer may redeem the notes early at its election")
show("effect of the 6.0% per annum daily deduction on the index")
show("Trigger Value 50.00% of the Initial Value", mode="keyword")


[hybrid] issuer may redeem the notes early at its election
  · [Callable Contingent Interest Notes Linked to the Lesser Performing of the Russell 2000® Index and the S&P 500
      ●THE OPTIONAL EARLY REDEMPTION FEATURE MAY FORCE A POTENTIAL EARLY EXIT — If we elect to redeem your notes early, the term of the notes may be reduced to as sho
  · [공통 문구 | JPMorgan Chase Financial 구조화 노트]
      ●THE OPTIONAL EARLY REDEMPTION FEATURE MAY FORCE A POTENTIAL EARLY EXIT — If we elect to redeem your notes early, the term of the notes may be reduced to as sho
  · [Callable Contingent Interest Notes Linked to the Least Performing of the Nasdaq-100 Index®, the Russell 2000®
      ●THE OPTIONAL EARLY REDEMPTION FEATURE MAY FORCE A POTENTIAL EARLY EXIT — If we elect to redeem your notes early, the term of the notes may be reduced to as sho

[hybrid] effect of the 6.0% per annum daily deduction on the index
  · [공통 문구 | JPMorgan Chase Financial 구조화 노트]
      ●THE LEVEL OF THE INDEX WILL INCLUDE A 6.0%

### 4-2. 한국어로 물으면?
영어 전용 모델이라 한국어 질문은 엉뚱한 결과를 냅니다. **이 한계는 Claude가 메워 줍니다.** MCP로 연결하면 Claude가 질문을 영어 검색어로 바꿔 도구를 호출합니다.

In [7]:
show("추정가치는 어떻게 산정되나요?")


[hybrid] 추정가치는 어떻게 산정되나요?
  · [공통 문구 | JPMorgan Chase Financial 구조화 노트]
      http://www.sec.gov/Archives/edgar/data/19617/000121390026045209/ea0285802-11_424b2.pdf
  · [공통 문구 | JPMorgan Chase Financial 구조화 노트]
      http://www.sec.gov/Archives/edgar/data/19617/000121390026045198/ea0285802-20_424b2.pdf
  · [공통 문구 | JPMorgan Chase Financial 구조화 노트]
      Price to Public (1) | Fees and Commissions (2)(3) | Proceeds to Issuer


### 4-3. RAG가 못 하는 질문

"배리어가 50%인 상품은?"이나 "오토콜 상품의 평균 쿠폰은?" 같은 **속성·집계 질문**을 검색으로 풀면 어떻게 될까요?

In [8]:
show("autocallable notes with a 50% trigger value", include_boilerplate=False, k=5)


[hybrid] autocallable notes with a 50% trigger value
  · [Auto Callable Contingent Interest Notes Linked to the Least Performing of the Nasdaq-100 Index®, the Russell 
      The table below illustrates the hypothetical total Contingent Interest Payments per $1,000 principal amount note over the term of the notes based on the Conting
  · [Auto Callable Contingent Interest Notes Linked to the Least Performing of the Nasdaq-100 Index®, the Russell 
      Because the closing level of each Index on the first Autocall Review Date, which is also the sixth Interest Review Date, is greater than or equal to its Initial
  · [Auto Callable Contingent Interest Notes Linked to the MerQube US Large-Cap Vol Advantage Index due September 
      The table below illustrates the hypothetical total Contingent Interest Payments per $1,000 principal amount note over the term of the notes based on the Conting
  · [Auto Callable Contingent Interest Notes Linked to the MerQube US Large-Cap Vol Advantage Index 

In [9]:
# 같은 질문을 SQL로
con.sql('''
    SELECT product_family, count(*) AS 상품수, round(avg(coupon_rate_pa) * 100, 2) AS 평균쿠폰_pct
    FROM products WHERE barrier_pct = 0.50 GROUP BY ALL ORDER BY 상품수 DESC
''').df()

,product_family,상품수,평균쿠폰_pct
0,autocall_contingent_coupon,8,16.1
1,autocall_call_premium,3,NaN
2,leveraged_barrier,2,NaN
3,issuer_call_contingent_coupon,1,9.0


| 질문 유형 | 예 | 맞는 도구 |
|---|---|---|
| 속성 필터·집계·비교 | "배리어 50% 상품 수", "상품군별 평균 쿠폰" | **SQL** (상품 DB) |
| 정의·의미·계산 방식 | "추정가치는 어떻게 산정?", "일일 차감의 영향은?" | **검색** (원문) |
| 특정 상품의 세부 조건 | "CUSIP 46661KGW2의 조기상환 조건" | **SQL로 찾고 원문으로 확인** |

검색은 문서 몇 개를 보여 줄 뿐 **세거나 평균을 내지 못합니다.** 반대로 SQL은 추출하지 않은 정보는 답하지 못합니다.
그래서 두 가지를 모두 도구로 주고, **어떤 도구를 쓸지는 Claude가 질문을 보고 고르게** 합니다. 이것이 MCP 연결의 핵심입니다.

In [10]:
con.close()  # MCP 서버가 DB를 읽을 수 있도록 연결을 닫는다 (DuckDB는 쓰기 연결이 열려 있으면 다른 프로세스가 못 엶)

## 5. MCP 서버로 Claude Desktop에 연결

`src/aifinance/mcp_server.py`는 네 가지 도구를 제공합니다.

| 도구 | 하는 일 | DB |
|---|---|---|
| `describe_database` | 테이블·컬럼·상품군·단위 규칙 안내 | SQL용 |
| `query_products` | 읽기 전용 SQL (SELECT/WITH 한 문장, 최대 200행) | SQL용 |
| `search_filings` | 원문 하이브리드 검색, 결과마다 EDGAR 링크 | 색인용 |
| `get_product` | CUSIP 하나의 조건·기초자산·공시 이력·품질 경고 | SQL용 |

- **SQL용 DB**: 모듈 3의 전체 기간 DB(`aifinance_full.duckdb`)가 있으면 그것, 없으면 표본 DB
- **색인용 DB**: 표본 DB. 정의·위험·산정 방식 설명은 상품군마다 공통이라 표본으로 충분하고, 전체를 색인하면 노트북 CPU로 1시간 이상 걸립니다.

서버의 `instructions`에 도구 선택 원칙(숫자는 SQL, 설명은 검색, 검색어는 영어, 출처 표기)을 넣어 두었습니다.

### 5-1. Code 탭에서 쓰기
저장소 루트의 `.mcp.json`에 서버가 등록돼 있습니다. Claude Desktop **Code 탭**으로 이 폴더를 열면 `aifinance` 서버를 쓸지 묻습니다. 허용하면 바로 사용할 수 있습니다.

### 5-2. 일반 채팅에서 쓰기
Claude Desktop → 설정 → 개발자 → 구성 편집으로 `claude_desktop_config.json`을 열고 아래 내용을 `mcpServers`에 추가한 뒤 Desktop을 재시작합니다. (아래 셀이 내 컴퓨터 경로로 만들어 줍니다.)

In [11]:
config = {"mcpServers": {"aifinance": {"command": sys.executable, "args": ["-m", "aifinance.mcp_server"]}}}
print(json.dumps(config, indent=2, ensure_ascii=False))

{
  "mcpServers": {
    "aifinance": {
      "command": "/Users/joocheol/Documents/GitHub/AIFinance/.venv/bin/python",
      "args": [
        "-m",
        "aifinance.mcp_server"
      ]
    }
  }
}


### 5-3. 연결 시험
Claude Desktop이 하는 것과 똑같이 서버를 띄우고 MCP 프로토콜로 도구 목록을 받아 호출해 봅니다.

In [12]:
from mcp.client.session import ClientSession
from mcp.client.stdio import StdioServerParameters, stdio_client


async def probe():
    params = StdioServerParameters(command=sys.executable, args=["-m", "aifinance.mcp_server"], cwd=str(PROJECT_ROOT))
    async with stdio_client(params) as (r, w):
        async with ClientSession(r, w) as s:
            await s.initialize()
            for t in (await s.list_tools()).tools:
                print(f"🔧 {t.name}: {(t.description or '').splitlines()[0]}")
            res = await s.call_tool("query_products", {"sql": "SELECT count(*) AS 상품수, min(pricing_date) AS 시작, max(pricing_date) AS 끝 FROM products"})
            print(res.content[0].text)

await probe()

🔧 describe_database: 상품 DB 의 테이블·컬럼 구조, 상품군 목록, 단위 규칙을 보여 준다. SQL 을 쓰기 전에 먼저 호출하라.
🔧 query_products: 상품 DB 에 읽기 전용 SQL(DuckDB 문법)을 실행한다. SELECT/WITH 한 문장만 허용, 최대 200행.
🔧 search_filings: 공시 원문을 하이브리드(의미 + 키워드) 검색한다. 검색어는 영어로 쓴다.
🔧 get_product: CUSIP 하나의 확정 조건, 예비본 조건, 기초자산, 공시 이력(EDGAR 링크)을 보여 준다.


|   상품수 | 시작                | 끝                  |
|---------:|:--------------------|:--------------------|
|     2971 | 2026-06-25 00:00:00 | 2026-11-09 00:00:00 |


## 🤖 Claude 실습 4

MCP 서버를 연결한 Claude Desktop(채팅 또는 Code 탭)에서 해 보세요. **Claude가 어떤 도구를 어떤 순서로 쓰는지** 눈여겨보세요.

**4-A. 도구 선택 관찰**
> JP Morgan 오토콜 조건부 쿠폰 상품의 평균 쿠폰과 배리어를 기초자산 개수(단일 vs 워스트오브 2개, 3개)별로 비교해줘. 그리고 워스트오브 구조에서 쿠폰이 왜 높아지는지 공시 원문의 위험 고지를 근거로 설명해줘.

**4-B. 원문으로 검증하기**
> 추정가치(estimated value)가 가장 낮은 상품 5개를 찾아, 각 상품의 공시 원문에서 추정가치 산정 문단과 판매 수수료를 확인해줘. 추정가치가 낮은 이유를 상품 구조와 연결해 설명하고, 출처 링크를 붙여줘.

**4-C. 검색 품질 개선** (Code 탭)
> `src/aifinance/rag.py`의 검색 품질을 평가할 질문 10개와 기대 정답 청크를 만들고, 청크 크기(800·1200·2000자)와 공통 문구 기준(3%·5%·10%)을 바꿔 가며 Recall@5를 비교해줘. 결과를 표로 보여주고 가장 좋은 설정을 추천해줘.

**4-D. 도구 추가** (Code 탭)
> `mcp_server.py`에 "비슷한 상품 찾기" 도구를 추가해줘. CUSIP을 받으면 같은 상품군에서 만기·배리어·쿠폰·기초자산이 가장 비슷한 상품 5개를 돌려주도록. 추가한 뒤 Claude Desktop을 재시작해 써 보고, 결과가 실무적으로 그럴듯한지 평가해줘.

---
**다음 모듈**: 모듈 5에서는 RAG 시스템을 **평가**합니다. 질문–정답 세트를 만들고, 검색 정확도와 Claude 답변의 사실성(근거와 일치하는지)을 측정합니다.